In [1]:
import json
import re
from yarn_utils import YARNGraph
from itertools import permutations

# Load Data

In [51]:
# FOLDER_PATH = "annotations/FRACAS_12032026/"
# FILE = "105h.yarn.json"

FOLDER_PATH = "annotations/"
FILE = "1.yarn.json"

In [52]:
with open(FOLDER_PATH + FILE) as f:
    yarn_graph_json = json.load(f)

yarn_graph = YARNGraph(yarn_graph_json)
yarn_grew = yarn_graph.grew()

In [53]:
with open('output.json', 'w') as f:
    json.dump(yarn_grew, f)

In [54]:
yarn_grew

{'nodes': {'vw1': {'pred': 'want-01', 'type': 'V', 'var': 'vw1'},
  'vb1': {'concept': 'boy', 'type': 'V', 'var': 'vb1'},
  'vj1': {'concept': 'John', 'type': 'V', 'var': 'vj1'},
  'vy1': {'concept': 'young', 'type': 'V', 'var': 'vy1'},
  'vd1': {'concept': 'dog', 'type': 'V', 'var': 'vd1'},
  'vl1': {'pred': 'large-01', 'type': 'V', 'var': 'vl1'},
  'vd2': {'concept': 'definitely', 'type': 'V', 'var': 'vd2'},
  'vb2': {'pred': 'bark-01', 'type': 'V', 'var': 'vb2'},
  's1': {'event': 's1', 'var': 's1', 'type': 'S'},
  's1-temp': {'type': 'F', 'feat': 'temp', 'var': 's1-temp'},
  's1-quant': {'type': 'F', 'feat': 'quant', 'var': 's1-quant'},
  's1-num': {'type': 'F', 'feat': 'num', 'var': 's1-num'},
  's1-def': {'type': 'F', 'feat': 'def', 'var': 's1-def'},
  's1-manner': {'type': 'F', 'feat': 'manner', 'var': 's1-manner'},
  's1-neg': {'type': 'F', 'feat': 'neg', 'var': 's1-neg'},
  's2': {'event': 's2', 'var': 's2', 'type': 'S'},
  's2-temp': {'type': 'F', 'feat': 'temp', 'var': 's2-t

# Preprocessing

In [55]:
# from grewpy import Graph, GRS

# grs_path = "grs/main.grs"
# grs = GRS(grs_path)
# yarn_grew = grs.apply(Graph(yarn_grew), strat='main')

In [56]:
# convert temp H edges to L edges

# Build F and R

In [57]:
variables = set()
def fresh_variable(base=None):
    if base is None:
        base = 'e'
    i = 0

    while True:
        variable = base if i == 0 else f"{base}{i}"
        if variable not in variables:
            variables.add(variable)
            break
        i += 1
    return variable

In [109]:
id2var = {}
F = []

for node, feats in yarn_grew['nodes'].items():

    # S, V -> S 
    # no S -> S for now
    if feats['type'] == 'S':

        id2var[node] = feats['var']

        for edge1 in yarn_grew['edges']:
            if edge1['tar'] == node:
                src = edge1['src']

                if yarn_grew['nodes'][src]['type'] == 'C':
                    for edge2 in yarn_grew['edges']:
                        if edge2['tar'] == src:
                            incoming_edge = edge2['src']
            else:
                incoming_edge = None

        F.append({
                        'id':node,
                        'incoming_edge': incoming_edge,
                        'outgoing_edge':None,
                        'type':"∃_s",
                        'variable': id2var[node],
                        'tar_label': 'S', # change to 'label'
                        #'definite': []
                    })
        
    # quantification
    if (feats['type'] == "L" or feats['type'] == 'H') and feats['feat'] in ['quant', 'temp']:
        for edge in yarn_grew['edges']:
            if edge['src'] == node:
                tar = edge['tar']

                if yarn_grew['nodes'][tar]['type'] == 'V':
                    
                    edge_label = feats['value'] if feats['value'] else feats['feat']

                    if 'pred' in yarn_grew['nodes'][tar]:
                        tar_label = yarn_grew['nodes'][tar]['pred']
                    else:
                        tar_label = yarn_grew['nodes'][tar]['concept']

                    # if feats['feat'] in ['quant', 'temp', 'aspect', 'neg', 'modal']:
                    #     if feats['feat'] in ['quant', 'temp']:
                    if tar not in id2var:
                        variable = fresh_variable(base=tar_label[0])
                        id2var[tar] = variable
                    else:
                        raise AssertionError(f"Variable for {tar} already exists in id2var")
                    
                    F.append({
                        'id':tar,
                        'incoming_edge':node,
                        'outgoing_edge':None,
                        'type':"Q_"+edge_label,
                        'variable': id2var[tar] if feats['feat'] in ['quant', 'temp'] else None,
                        'tar_label':tar_label, # change to 'label'
                        #'definite': []
                    })
    
    # negation, modality, aspect
    if (feats['type'] == "L" or feats['type'] == 'H') and feats['feat'] in ['neg', 'modal', 'aspect']:
        for edge1 in yarn_grew['edges']:
            if edge1['src'] == node:
                tar = edge1['tar']

                if yarn_grew['nodes'][tar]['type'] in ['V', 'L', 'H']:
                    
                    edge_label = feats['value'] if feats['value'] else feats['feat']

                    for edge2 in yarn_grew['edges']:
                        if edge2['tar'] == node:
                            src = edge2['src']

                    F.append({
                        'id':node,
                        'incoming_edge':src,
                        'outgoing_edge':tar,
                        'type':"Q_"+edge_label,
                        'variable': None,
                        'tar_label': None, # change to 'label'
                        #'definite': []
                    })
                
                # if yarn_grew['nodes'][tar]['type'] == 'L' or yarn_grew['nodes'][tar]['type'] == 'H':
                #     edge_label = feats['value'] if feats['value'] else feats['feat']

                #     for edge2 in yarn_grew['edges']:
                #         if edge2['tar'] == node:
                #             src = edge2['src']

                #     F.append({
                #         'id':node,
                #         'incoming_edge':src,
                #         'outgoing_edge':tar,
                #         'type':"Q_"+edge_label,
                #         'variable': None,
                #         'tar_label': None, # change to 'label'
                #         'definite': []
                #     })

    # L/H -> V
    # # the logical stuff?
    # if feats['type'] == "L" or feats['type'] == 'H':
    #     for edge in yarn_grew['edges']:
    #         if edge['src'] == node:
    #             tar = edge['tar']

    #             if yarn_grew['nodes'][tar]['type'] == 'V':
    #                 edge_label = feats['value'] if feats['value'] else feats['feat']
    #                 if 'pred' in yarn_grew['nodes'][tar]:
    #                     tar_label = yarn_grew['nodes'][tar]['pred']
    #                 else:
    #                     tar_label = yarn_grew['nodes'][tar]['concept']

    #                 if tar not in id2var:
    #                     variable = fresh_variable(base=tar_label[0])
    #                     id2var[tar] = variable

    #                 F.append({
    #                     'id':tar,
    #                     'incoming_edge':node,
    #                     'type':"Q_"+edge_label,
    #                     'variable': id2var[tar],
    #                     'tar_label':tar_label, # change to 'label'
    #                     'definite': []
    #                 })
    
    # H -> L/H
    # if feats['type'] == 'H':
    #     for edge in yarn_grew['edges']:
    #         if edge['src'] == node:
    #             tar = edge['tar']

    #             if yarn_grew['nodes'][tar]['type'] == 'L' or yarn_grew['nodes'][tar]['type'] == 'H':
    #                 edge_label = feats['value'] if feats['value'] else feats['feat']

    #                 F.append({
    #                     'id':tar,
    #                     'incoming_edge':node,
    #                     'type':"Q_"+edge_label,
    #                     'variable': None,
    #                     'tar_label': None, # change to 'label'
    #                     'definite': []
    #                 })

# Ignore definitness and number for now
F = [f for f in F if f['type'] not in ['Q_indefinite', 'Q_singular', 'Q_plural', 'Q_manner', 'Q_perfective']]

In [110]:
# # Create R by getting all relations between V nodes

# relations = set()
# for node, feats in yarn_grew['nodes'].items():
#     if feats['type'] == "E":
#         edge_label = feats['rel']

#         for edge in yarn_grew['edges']:
#             if edge['tar'] == node:
#                 src = edge['src']
                
#             if edge['src'] == node:
#                 tar = edge['tar']
                

#         relations.add((edge_label, src, tar))
    
        

# R = {}
# for relation in relations:
#     label = relation[0]
#     src = relation[1]
#     tar = relation[2]

#     if src not in R:
#         R[src] = [(label, src, tar)]
#     else:
#         R[src].append((label, src, tar))

In [ ]:
R = {}
for node, feats in yarn_grew['nodes'].items():
    if feats['type'] == "E":
        edge_label = feats['rel']

        for edge1 in yarn_grew['edges']:
            if edge1['tar'] == node:
                src = edge1['src']

                for edge2 in yarn_grew['edges']:
                    if edge2['src'] == node:
                        tar = edge2['tar']
                
                        if src not in R:
                            R[src] = [(edge_label, src, tar)] # convert to string immediately
                        else:
                            R[src].append((edge_label, src, tar)) # convert to string immediately
    
    if feats['type'] == "C":
        edge_label = feats['rel']

        for edge1 in yarn_grew['edges']:
            if edge1['tar'] == node:
                src = edge1['src']

                for edge2 in yarn_grew['edges']:
                    if edge2['src'] == node:
                        tar = edge2['tar']
                
                        if tar not in R:
                            R[tar] = [(edge_label, src, tar)] # convert to string immediately
                        else:
                            R[tar].append((edge_label, src, tar)) # convert to string immediately
    

    if feats['type'] == "L" and feats['feat'] == 'def': 
        for edge1 in yarn_grew['edges']:
            if edge1['src'] == node:
                tar = edge1['tar']

                if yarn_grew['nodes'][tar]['type'] == 'V':
                    
                    if tar not in R:
                        R[tar] = [f'C({id2var[tar]})'] #treat as a constant?
                    else:
                        R[tar].append(f'C({id2var[tar]})')
    
    if feats['type'] == "L" and feats['feat'] == 'num' and feats['value'] == 'plural': 
        for edge1 in yarn_grew['edges']:
            if edge1['src'] == node:
                tar = edge1['tar']

                if yarn_grew['nodes'][tar]['type'] == 'V':
                    
                    if tar not in R:
                        R[tar] = [f'plural({id2var[tar]})'] #treat as a constant?
                    else:
                        R[tar].append(f'plural({id2var[tar]})')

    # S node modifiers
    # update as you annotate if you find trickier cases
        
    if feats['type'] == "L" and feats['feat'] in ['manner', 'loc', 'dir', 'duration', 'mod', 'freq']: 
        for edge1 in yarn_grew['edges']:
            if edge1['src'] == node:
                tar = edge1['tar']

                if yarn_grew['nodes'][tar]['type'] == 'V':
                    
                    edge_label = feats['value'] if feats['value'] else feats['feat']

                    if 'pred' in yarn_grew['nodes'][tar]:
                        tar_label = yarn_grew['nodes'][tar]['pred']
                    else:
                        tar_label = yarn_grew['nodes'][tar]['concept']

                    for edge2 in yarn_grew['edges']:
                        if edge2['tar'] == node:
                            src = edge2['src'].split('-')[0]
                        
                            if src not in R:
                                R[src] = [(edge_label, src, tar_label.upper())] #treat as a constant?
                            else:
                                R[src].append((edge_label, src, tar_label.upper()))

In [112]:
# Deal with definiteness
# Q_exists and Q_forall only for now

to_delete = []
for f1 in F:
    for f2 in F:
        if f1['type'] == 'Q_definite' and (f2['type'] == 'Q_exists' or f2['type'] == 'Q_forall') and f1['id'] == f2['id']:
            f2['definite'] = [f'C({f2['variable']})']
            to_delete.append(f1)

for item in to_delete:
    if item in F:
        F.remove(item)

In [113]:
F

[{'id': 's1',
  'incoming_edge': None,
  'outgoing_edge': None,
  'type': '∃_s',
  'variable': 's1',
  'tar_label': 'S'},
 {'id': 's2',
  'incoming_edge': 'vw1',
  'outgoing_edge': None,
  'type': '∃_s',
  'variable': 's2',
  'tar_label': 'S'},
 {'id': 'vw1',
  'incoming_edge': 'l1',
  'outgoing_edge': None,
  'type': 'Q_present',
  'variable': 'w1',
  'tar_label': 'want-01'},
 {'id': 'vb1',
  'incoming_edge': 'l2',
  'outgoing_edge': None,
  'type': 'Q_forall',
  'variable': 'b2',
  'tar_label': 'boy'},
 {'id': 'vb2',
  'incoming_edge': 'l6',
  'outgoing_edge': None,
  'type': 'Q_present',
  'variable': 'b3',
  'tar_label': 'bark-01'},
 {'id': 'vd1',
  'incoming_edge': 'l7',
  'outgoing_edge': None,
  'type': 'Q_exists',
  'variable': 'd1',
  'tar_label': 'dog'},
 {'id': 'l10',
  'incoming_edge': 's1-neg',
  'outgoing_edge': 'vw1',
  'type': 'Q_neg',
  'variable': None,
  'tar_label': None}]

In [114]:
R

{'vw1': [('ARG0', 'vw1', 'vb1')],
 'vb1': [('mod', 'vb1', 'vy1'), ('name', 'vb1', 'vj1'), 'C(b2)'],
 'vb2': [('ARG0', 'vb2', 'vd1')],
 'vl1': [('ARG1', 'vl1', 'vb2')],
 's1': [('manner', 's1', 'DEFINITELY')],
 'vd1': ['C(d1)'],
 's2': [('ARG1', 'vw1', 's2')]}

# Create the Forest

In [108]:
forest = {'nodes':{}, 'edges':[]}

for i, f in enumerate(F):
    forest['nodes'][i] = {
            'id':f['id'],
            'incoming_edge':f['incoming_edge'],
            'type': f['type'],
            'variable': f['variable'],
            'tar_label': f['tar_label'],
            'relations': [f"{rel[0]}({f['variable']},{id2var[rel[2]]})" for rel in R[f['id']]] if f['id'] in R else [],
            'definite': f['definite']} # integrate R



for k1, v1 in forest['nodes'].items(): # encode specified scope
    for k2, v2 in forest['nodes'].items():
        if v1['id'] == v2['incoming_edge']:
            forest['edges'].append({'src':k1, 'rel':'', 'tar':k2})

KeyError: 'DEFINITELY'

In [ ]:
forest

{'nodes': {0: {'id': 'vf1',
   'incoming_edge': 'l1',
   'type': 'Q_past',
   'variable': 'f2',
   'tar_label': 'finish-01',
   'relations': ['ARG0(f2,d2)', 'ARG1(f2,r2)'],
   'definite': []},
  1: {'id': 'vd1',
   'incoming_edge': 'l3',
   'type': 'Q_exists',
   'variable': 'd2',
   'tar_label': 'delegate',
   'relations': [],
   'definite': []},
  2: {'id': 'vr1',
   'incoming_edge': 'l8',
   'type': 'Q_exists',
   'variable': 'r2',
   'tar_label': 'report',
   'relations': [],
   'definite': ['C(r2)']},
  3: {'id': 'l3',
   'incoming_edge': 'h1',
   'type': 'Q_neg',
   'variable': None,
   'tar_label': None,
   'relations': [],
   'definite': []}},
 'edges': [{'src': 3, 'rel': '', 'tar': 1}]}

## Add the Participant before Event constraint

In [ ]:
# C (predicates are introduced after their arguments)
forest_c = forest.copy()
for src, rels in R.items():
    for rel in rels:
        tar = rel[2]

        for k1, v1 in forest_c['nodes'].items():
            for k2, v2 in forest_c['nodes'].items():
                if v1['id'] == src and v2['id'] == tar:
                    forest_c['edges'].append({'src':k2, 'rel':None, 'tar':k1})

In [ ]:
forest_c

{'nodes': {0: {'id': 'vf1',
   'incoming_edge': 'l1',
   'type': 'Q_past',
   'variable': 'f2',
   'tar_label': 'finish-01',
   'relations': ['ARG0(f2,d2)', 'ARG1(f2,r2)'],
   'definite': []},
  1: {'id': 'vd1',
   'incoming_edge': 'l3',
   'type': 'Q_exists',
   'variable': 'd2',
   'tar_label': 'delegate',
   'relations': [],
   'definite': []},
  2: {'id': 'vr1',
   'incoming_edge': 'l8',
   'type': 'Q_exists',
   'variable': 'r2',
   'tar_label': 'report',
   'relations': [],
   'definite': ['C(r2)']},
  3: {'id': 'l3',
   'incoming_edge': 'h1',
   'type': 'Q_neg',
   'variable': None,
   'tar_label': None,
   'relations': [],
   'definite': []}},
 'edges': [{'src': 3, 'rel': '', 'tar': 1},
  {'src': 1, 'rel': None, 'tar': 0},
  {'src': 2, 'rel': None, 'tar': 0}]}

# Get All Possible Trees

In [ ]:
import itertools
import networkx as nx

def get_all_possible_trees(n):
    nodes = list(range(n))
    for seq in itertools.product(nodes, repeat=n-2):
        yield nx.from_prufer_sequence(seq)

In [ ]:
nodes = list(forest_c['nodes'].keys())
n_nodes = len(nodes)

all_possible_trees = []

for tree in get_all_possible_trees(n_nodes):

    for root in nodes:

        visited = set([root])
        stack = [root]
        directed_edges = []

        while stack:
            current = stack.pop()

            for neighbor in tree.neighbors(current):
                if neighbor not in visited:
                    visited.add(neighbor)
                    stack.append(neighbor)

                    directed_edges.append({'src': current,'tar': neighbor})

        all_possible_trees.append({'edges': directed_edges})

In [ ]:
all_possible_trees

[{'edges': [{'src': 0, 'tar': 1}, {'src': 0, 'tar': 2}, {'src': 0, 'tar': 3}]},
 {'edges': [{'src': 1, 'tar': 0}, {'src': 0, 'tar': 2}, {'src': 0, 'tar': 3}]},
 {'edges': [{'src': 2, 'tar': 0}, {'src': 0, 'tar': 1}, {'src': 0, 'tar': 3}]},
 {'edges': [{'src': 3, 'tar': 0}, {'src': 0, 'tar': 1}, {'src': 0, 'tar': 2}]},
 {'edges': [{'src': 0, 'tar': 2}, {'src': 0, 'tar': 1}, {'src': 1, 'tar': 3}]},
 {'edges': [{'src': 1, 'tar': 0}, {'src': 1, 'tar': 3}, {'src': 0, 'tar': 2}]},
 {'edges': [{'src': 2, 'tar': 0}, {'src': 0, 'tar': 1}, {'src': 1, 'tar': 3}]},
 {'edges': [{'src': 3, 'tar': 1}, {'src': 1, 'tar': 0}, {'src': 0, 'tar': 2}]},
 {'edges': [{'src': 0, 'tar': 1}, {'src': 0, 'tar': 2}, {'src': 2, 'tar': 3}]},
 {'edges': [{'src': 1, 'tar': 0}, {'src': 0, 'tar': 2}, {'src': 2, 'tar': 3}]},
 {'edges': [{'src': 2, 'tar': 0}, {'src': 2, 'tar': 3}, {'src': 0, 'tar': 1}]},
 {'edges': [{'src': 3, 'tar': 2}, {'src': 2, 'tar': 0}, {'src': 0, 'tar': 1}]},
 {'edges': [{'src': 0, 'tar': 1}, {'src'

In [ ]:
len(all_possible_trees)

64

# Build T_all

In [ ]:
# Gets the children of nodes that don't introduce variables
def get_H_children(graph):
    H_children_dict = {}
    for edge in graph['edges']:
        src = edge['src']
        tar = edge['tar']
        if not graph['nodes'][src]['variable']:
            H_children_dict[src] = H_children_dict.get(src, []) + [tar]

    return H_children_dict

In [ ]:
# Get all children
def get_children(graph):
    children_dict = {}
    for edge in graph['edges']:
        src = edge['src']
        tar = edge['tar']
        children_dict[src] = children_dict.get(src, []) + [tar]
    return children_dict

# Extend children to descendants
def get_descendants(node, children_dict):
    descendants = []
    for child in children_dict.get(node, []):
        descendants.append(child)
        descendants.extend(get_descendants(child, children_dict))

    return descendants

def get_all_descendants(graph):
    children_dict = get_children(graph)
    descendants_dict = {}
    for node in children_dict:
        descendants_dict[node] = get_descendants(node, children_dict)

    return descendants_dict

In [ ]:
# Constraint Checkers

def check_compatibility_of_scopes(tree, forest):
    descendants_tree = get_all_descendants(tree)
    descendants_forest = get_all_descendants(forest)

    for k,v in descendants_forest.items():
        for descendant in v:
            if k in descendants_tree:
                if descendant not in descendants_tree[k]:
                    return False
            else:
                return False
    return True

def check_locality_of_features(tree, forest):
    children_tree = get_children(tree)
    H_children_forest = get_H_children(forest)
    
    for k,v in H_children_forest.items():
        for child in v:
            if k in children_tree:
                if child not in children_tree[k]:
                    return False
            else:
                return False
    return True

In [ ]:
descendants_forest = get_all_descendants(forest)
H_children_forest = get_H_children(forest)
print(descendants_forest)
print(H_children_forest)

{3: [1, 0], 1: [0], 2: [0]}
{3: [1]}


In [ ]:
valid_trees = [tree for tree in all_possible_trees if check_locality_of_features(tree, forest_c)]
valid_trees = [tree for tree in valid_trees if check_compatibility_of_scopes(tree, forest_c)]
valid_trees

[{'edges': [{'src': 2, 'tar': 3}, {'src': 3, 'tar': 1}, {'src': 1, 'tar': 0}]},
 {'edges': [{'src': 3, 'tar': 1}, {'src': 1, 'tar': 2}, {'src': 2, 'tar': 0}]}]

In [ ]:
T_all = []
for tree in valid_trees:
    new_tree = forest_c.copy()
    new_tree['edges'] = tree['edges']
    T_all.append(new_tree)

## Reformat T_all

In [ ]:
# Reformat to linear tree for easier interpretation

def graph_to_linear_tree(graph):
    nodes = graph["nodes"]
    edges = graph["edges"]

    next_node = {}
    for e in edges:
        next_node[e["src"]] = e["tar"]

    all_nodes = set(nodes.keys())
    all_targets = {e["tar"] for e in edges}
    root = (all_nodes - all_targets).pop()

    def build(node_id):
        node_data = dict(nodes[node_id])

        if node_id in next_node:
            node_data["child"] = build(next_node[node_id])
        else:
            node_data["child"] = None

        return node_data

    return build(root)

In [ ]:
T_all = [graph_to_linear_tree(tree) for tree in T_all]

In [ ]:
T_all[0]

{'id': 'vr1',
 'incoming_edge': 'l8',
 'type': 'Q_exists',
 'variable': 'r2',
 'tar_label': 'report',
 'relations': [],
 'definite': ['C(r2)'],
 'child': {'id': 'l3',
  'incoming_edge': 'h1',
  'type': 'Q_neg',
  'variable': None,
  'tar_label': None,
  'relations': [],
  'definite': [],
  'child': {'id': 'vd1',
   'incoming_edge': 'l3',
   'type': 'Q_exists',
   'variable': 'd2',
   'tar_label': 'delegate',
   'relations': [],
   'definite': [],
   'child': {'id': 'vf1',
    'incoming_edge': 'l1',
    'type': 'Q_past',
    'variable': 'f2',
    'tar_label': 'finish-01',
    'relations': ['ARG0(f2,d2)', 'ARG1(f2,r2)'],
    'definite': [],
    'child': None}}}}

# Interpretation

In [ ]:
def interpret(root, temp_variable, colors = False):
     
    if root is None:
         return ""
     
    if root['variable']:
     
        if root["type"] == "Q_exists":
            return "∃" + root["variable"] + ". (" + root["tar_label"] + "(" + root['variable'] + ") ∧ " + \
        " ∧ ".join(root['definite']) + " ∧ " + " ∧ ".join(root['relations']) + "(" + interpret(root['child'], temp_variable) + ")"

        if root["type"] == "Q_forall":
            return "∀" + root["variable"] + ". (" + root["tar_label"] + "(" + root['variable'] + ") → " + \
        " ∧ ".join(root['definite']) + " ∧ " + " ∧ ".join(root['relations']) + "(" + interpret(root['child'], temp_variable) + ")"

        if root["type"] == "Q_present":
            return "∃" + root["variable"] + ". (" + root["tar_label"] + "(" + root['variable'] + ") ∧ " + \
        root['variable'] + "O" + temp_variable + " ∧ " + \
        " ∧ ".join(root['relations']) + "(" + interpret(root['child'], root['variable']) + ")"

        if root["type"] == "Q_past":
            return "∃" + root["variable"] + ". (" + root["tar_label"] + "(" + root['variable'] + ") ∧ " + \
        root['variable'] + "≺" + temp_variable + " ∧ " + \
        " ∧ ".join(root['relations']) + "(" + interpret(root['child'], root['variable']) + ")"

        if root["type"] == "Q_future":
            return "∃" + root["variable"] + ". (" + root["tar_label"] + "(" + root['variable'] + ") ∧ " + \
        temp_variable + "≺" + root['variable'] + " ∧ " + \
        " ∧ ".join(root['relations']) + "(" + interpret(root['child'], root['variable']) + ")"

        if root["type"] == "Q_possibility":
            return "◇" + " (" + "∃" + root["variable"] + ". (" + root["tar_label"] + "(" + root['variable'] + ") ∧ " + \
        " ∧ ".join(root['relations']) + interpret(root["child"], temp_variable) + ")" + ")"

        if root["type"] == "Q_necessity":
            return "□" + " (" + "∃" + root["variable"] + ". (" + root["tar_label"] + "(" + root['variable'] + ") ∧ " + \
        " ∧ ".join(root['relations']) + interpret(root["child"], temp_variable) + ")" + ")"

        if root["type"] == "Q_neg":
            return "¬" + " (" + "∃" + root["variable"] + ". (" + root["tar_label"] + "(" + root['variable'] + ") ∧ " + \
        " ∧ ".join(root['relations']) + interpret(root["child"], temp_variable) + ")" + ")"
    
    else:

        if root["type"] == "Q_neg":
            return "¬" + " (" + interpret(root["child"], temp_variable) + ")"
        
        if root["type"] == "Q_possibility":
            return "◇" + " (" + interpret(root["child"], temp_variable) + ")"
        
        if root["type"] == "Q_necessity":
            return "□" + " (" + interpret(root["child"], temp_variable) + ")"

        if root["type"] == "Q_present":
            return "∃" + fresh_variable() + ". (" + root['variable'] + "O" + temp_variable + " ∧ " + \
        "(" + interpret(root['child'], root['variable']) + ")"

        if root["type"] == "Q_past":
            return "∃" + fresh_variable() + ". (" + root['variable'] + "≺" + temp_variable + " ∧ " + \
        "(" + interpret(root['child'], root['variable']) + ")"

        if root["type"] == "Q_future":
            return "∃" + fresh_variable() + ". (" + temp_variable + "≺" + root['variable'] + " ∧ " + \
        "(" + interpret(root['child'], root['variable']) + ")"
     
def clean_formula(formula):
    clean_formula = formula.replace("()", "")
    clean_formula = clean_formula.replace("∧  ∧", "∧")
    return clean_formula

In [ ]:
for tree in T_all:
    print(clean_formula(interpret(tree, "now")))

∃r2. (report(r2) ∧ C(r2) ∧ (¬ (∃d2. (delegate(d2) ∧ (∃f2. (finish-01(f2) ∧ f2≺now ∧ ARG0(f2,d2) ∧ ARG1(f2,r2))))
¬ (∃d2. (delegate(d2) ∧ (∃r2. (report(r2) ∧ C(r2) ∧ (∃f2. (finish-01(f2) ∧ f2≺now ∧ ARG0(f2,d2) ∧ ARG1(f2,r2))))
